# NB03: Data Analysis

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup
Import packages

In [35]:
import pandas as pd
import plotly.express as px
import numpy as np
import statsmodels.formula.api as smf

Declare constants

In [ ]:
# Month of ChatGPT launch
T_0 = pd.Timestamp("2022-11-01")
T_0

Timestamp('2022-11-01 00:00:00')

Load long form dataframe and extract key values for analysis:
- Most recent month with data for all industries

In [4]:
df = pd.read_csv("../data/processed/employment.csv", parse_dates=["date"])
industry_codes = pd.read_csv("../data/reference/industry_code_map.csv")

# Group by industry and most recent date, then take the minimum date
t_end = df.groupby("industry_name")["date"].max().min()

# Number of unique industries in sample
unique_industries = df.naics.nunique()

df.head()

,date,industry_name,industry_code,naics,ai_exposure,employment,sector
0,2026-06-01,Offices of physicians,65621100,6211,1.009907,3053.7,Private education and health services
1,2026-05-01,Offices of physicians,65621100,6211,1.009907,3054.6,Private education and health services
2,2026-04-01,Offices of physicians,65621100,6211,1.009907,3050.7,Private education and health services
3,2026-03-01,Offices of physicians,65621100,6211,1.009907,3049.8,Private education and health services
4,2026-02-01,Offices of physicians,65621100,6211,1.009907,3011.8,Private education and health services


## Data coverage
As I was unable to match a significant number of AIIE scores to BLS series, I check the coverage of my dataset against the expanded out list of AIIE scores.

In [ ]:

# Extract the number of covered industries and total employment for each sector
covered = (df[df["date"] == t_end]
    .groupby("sector")
    .agg(industries=("industry_name", "size"),
    employment=("employment", "sum"))
)

# Calculate total number of series in each sector


# Get unique 3-digit aggregate series IDs
agg_codes = df.loc[df["industry_code"].astype(str).str[5] == "0", "industry_code"].unique().tolist()
agg_codes

prefixes = [str(x)[:-3] for x in agg_codes]

prefixes

industry_codes = industry_codes.assign(
    digit_3 = industry_codes["industry_code"].astype(str).str[:-3],
    )
industry_codes.loc[industry_codes["3_dig"].isin(prefixes)]



,industry_code,naics,industry_name,3_dig
24,31327000,327,Nonmetallic mineral product manufacturing,31327
25,31327100,3271,Clay product and refractory manufacturing,31327
26,31327200,3272,Glass and glass product manufacturing,31327
27,31327300,3273,Cement and concrete product manufacturing,31327
160,43484000,484,Truck transportation,43484
161,43484100,4841,General freight trucking,43484
162,43484200,4842,Specialized freight trucking,43484
187,50517000,517,Telecommunications,50517
188,50517100,5171,Wired and wireless telecommunications (except ...,50517
199,55531000,531,Real estate,55531


In [19]:
ss_df =  pd.json_normalize(
    ss_json["Results"]["series"], 
    record_path="data",
    meta="seriesID")

ss_df = (ss_df
    .assign(
        year=ss_df.year.astype(int),
        month=ss_df.period.str[1:].astype(int),
        industry=ss_df.seriesID.str[3:-2])
    .query(f"year == {t_end.year} & month == {t_end.month}")
)

ss_df = ss_df[["industry", "value"]]
# aiie = pd.read_csv("../data/reference/aiie.csv")
# aiie["naics_2"] = aiie["naics"].astype(str).str[:2]

## Employment growth vs AI exposure
Create plot_df for each chart I want to use for analysis. Merge required tables from NB02

In [33]:
first = (
    df.query(f"date == @T_0") # @ used to bring variable into query string
    .groupby("industry_name")
    .first()[["employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_start"})
    )

last = (
    df.query(f"date == @t_end") 
    .groupby("industry_name")
    .first()[["employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_end"})
    )

plot_df = (
    pd.merge(
        left=first, 
        right=last, 
        on=["industry_name", "sector", "ai_exposure"], 
        how="outer")
        .assign(
            emp_growth_pct=lambda x:
            (x.emp_end - x.emp_start) / x.emp_start *100
            )
        .reset_index()
)

fig = px.scatter(
    plot_df,
    x="ai_exposure",
    y="emp_growth_pct",
    color="sector",
    size="emp_start",
    trendline="ols",
    trendline_scope="overall", # Stop plotly from adding a trendline per
    title="Employment Growth vs AI Exposure"
)

fig.show()


In [ ]:
plot_df = (
    df
    .query()
    .pipe()
    .groupby
    .apply()
)

px.bar(plot_df, x, y)

TypeError: DataFrame.query() missing 1 required positional argument: 'expr'

## Sensitivity analysis
With and without 44 and 45 codes: 